# **Ejemplo de GPU Escala de Grises**

Este ejercicio se ha optado para aplicar 2 multidimensiones en GPU. El ejemplo modifica el color de una imagen parametrizada a escala de grises. El cálculo de la escala se realiza convirtiendo los 3 canales RGB, que representan a los colores R (*rojo*)-G(*verde*)-B(*azul*), de cada pixel siguiendo la ecuación:

<center>$ Pixel=R*0.30+G*0.59+B*0.11$</center>

EL objetivo es enseñar el funcionamiento del Lenguaje Python, CUDA y el manejo de imágenes a bajo nivel. El ejemplo es ilustrativo, ya que internamente el módulo Pillow posee varios filtros integrados.



---
# 2 Armado del ambiente
Toma la dirección web de una imagen con acceso público en internet, la deja disponible al contexto de ejecución del cuaderno Colab.

In [1]:
#@title # 2.1 Parámetros de ejecución
#@markdown ---
#@markdown ### Especifique la URL de la imagen:
url_imagen = "https://raw.githubusercontent.com/soa-pc-unlam/ProgramacionConcurrente/main/Enunciados%20TPs/TP-GPU/MesiCopa.jpg" #@param {type:"string"}

#@markdown ---
# Leo la imagen desde internet.
#!wget https://github.com/wvaliente/SOA_HPC/blob/main/unlam.jpg?raw=true -O imagen.jpg

# TODO: Mejorar informaciòn y resutlado de ejecución.
!wget {url_imagen} -O imagen.jpg




[0] Downloading 'https://raw.githubusercontent.com/soa-pc-unlam/ProgramacionConcurrente/main/Enunciados%20TPs/TP-GPU/MesiCopa.jpg' ...
Saving 'imagen.jpg'
HTTP response 200  [https://raw.githubusercontent.com/soa-pc-unlam/ProgramacionConcurrente/main/Enunciados%20TPs/TP-GPU/MesiCopa.jpg]


---
## 2.2 Instala en el cuaderno el módulo CUDA de Python.

In [2]:
!pip install pycuda

  Using cached pycuda-2026.1.tar.gz (1.7 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pytools-2026.1.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached mako-1.4.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached siphash24-1.9-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
Using cached pytools-2026.1.1-py3-none-any.whl (99 kB)
Using cached siphash24-1.9-cp314-cp314-win_amd64.whl (79 kB)
Using cached mako-1.4.3-py3-none-any.whl (80 kB)
Failed to build pycuda


  error: subprocess-exited-with-error
  
  × Building wheel for pycuda (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [239 lines of output]
      ***************************************************************
      *** WARNING: nvcc not in path.
      *** May need to set CUDA_INC_DIR for installation to succeed.
      ***************************************************************
      *************************************************************
      *** I have detected that you have not run configure.py.
      *************************************************************
      *** Additionally, no global config files were found.
      *** I will go ahead with the default configuration.
      *** In all likelihood, this will not work out.
      ***
      *** See README_SETUP.txt for more information.
      ***
      *** If the build does fail, just re-run configure.py with the
      *** correct arguments, and then retry. Good luck!
      *************************

---
# 3 Desarrollo
Ejecución del algoritmo escala de grises en GPU.

In [3]:
#Comentar esta linea para mostrar las imagenes
#%%writefile filter_image.py

#Comentar esta linea para medir los tiempos
%matplotlib inline

import matplotlib.pyplot as plt
import numpy
from PIL import Image
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# -----------------------------------------------------------------------------
img_nombre = 'imagen.jpg'
image = Image.open( img_nombre )

# summarize some details about the image
img_ancho, img_alto  = image.size

# -----------------------------------------------------------------------------
img_O_cpu = numpy.asarray(image)
img_O_cpu = img_O_cpu.astype( numpy.int32() )
img_R_cpu = numpy.empty_like( img_O_cpu)

img_O_gpu = cuda.mem_alloc( img_O_cpu.nbytes )
img_R_gpu = cuda.mem_alloc( img_R_cpu.nbytes )

cuda.memcpy_htod( img_O_gpu, img_O_cpu )
cuda.memcpy_htod( img_R_gpu, img_R_cpu )

# -----------------------------------------------------------------------------

module = SourceModule("""
__global__ void kernel_img(int ancho, int alto, int *img_O, int *img_R)
{
  int idx = threadIdx.x + blockIdx.x * blockDim.x;
  int idy = threadIdx.y + blockIdx.y * blockDim.y;

  if (idx < ancho && idy < alto)
  {
    int pos_pixel = ((idy * ancho) + idx) * 3;

    float fGris = 0.0;
    fGris += img_O[pos_pixel]     * 0.30;
    fGris += img_O[pos_pixel + 1] * 0.59;
    fGris += img_O[pos_pixel + 2] * 0.11;

    int gris = (int)fGris;

    img_R[pos_pixel]     = gris;
    img_R[pos_pixel + 1] = gris;
    img_R[pos_pixel + 2] = gris;
  }
}
""")

# -----------------------------------------------------------------------------
kernel = module.get_function("kernel_img")

dim_hilo_x = 6
dim_bloque_x = int( (img_ancho+dim_hilo_x-1) / dim_hilo_x )

dim_hilo_y = 6
dim_bloque_y = int( (img_alto+dim_hilo_y-1) / dim_hilo_y )

# -----------------------------------------------------------------------------
kernel( numpy.int32(img_ancho), numpy.int32(img_alto), img_O_gpu, img_R_gpu,
        block=( dim_hilo_x, dim_hilo_y, 1 ),
        grid=(dim_bloque_x, dim_bloque_y,1) )



# -----------------------------------------------------------------------------
cuda.memcpy_dtoh( img_R_cpu, img_R_gpu )

img_O_gpu.free()
img_R_gpu.free()

# -----------------------------------------------------------------------------
# Muestro los atributos de la imagen y como se ve antes del seudo filtro.
print("Imagen del filtro: ", img_nombre , " - tipo " , image.mode , "- [" , img_ancho , ", " , img_alto , "]" )
# -----------------------------------------------------------------------------
# Muestro los atributos de la imagen y como se ve antes del seudo filtro.
print("Imagen del filtro: ", img_nombre , " - tipo " , image.mode , "- [" , img_ancho , ", " , img_alto , "]" )
print( "Grilla : [", dim_bloque_x, ",", dim_bloque_y, "], Bloques: [", dim_hilo_x, ",", dim_hilo_y, " ] "  )


# -----------------------------------------------------------------------------
# Muestro la imagen Original el filtro.
plt.figure()
imgplot=plt.imshow( img_O_cpu )

# Muestro la imagen luego de aplicarle el filtro.
plt.figure()
imgplot=plt.imshow( img_R_cpu )



ModuleNotFoundError: No module named 'pycuda'

## 3.1 Ejecución y profiling del script

Uitlizando nvprof se ejecuta el script y se miden los tiempos de ejecución.

In [ ]:
!nvprof python filter_image.py

==2764== NVPROF is profiling process 2764, command: python3 filter_image.py
Imagen del filtro:  imagen.jpg  - tipo  RGB - [ 1024 ,  768 ]
Imagen del filtro:  imagen.jpg  - tipo  RGB - [ 1024 ,  768 ]
Grilla : [ 171 , 128 ], Bloques: [ 6 , 6  ] 
==2764== Profiling application: python3 filter_image.py
==2764== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   65.65%  7.3472ms         1  7.3472ms  7.3472ms  7.3472ms  [CUDA memcpy DtoH]
                   32.19%  3.6032ms         2  1.8016ms  1.7669ms  1.8363ms  [CUDA memcpy HtoD]
                    2.16%  241.88us         1  241.88us  241.88us  241.88us  kernel_img
      API calls:   59.87%  181.95ms         1  181.95ms  181.95ms  181.95ms  cuCtxCreate
                   34.97%  106.28ms         1  106.28ms  106.28ms  106.28ms  cuCtxDetach
                    3.14%  9.5483ms         1  9.5483ms  9.5483ms  9.5483ms  cuMemcpyDtoH
                    1.35%  4.1126ms       

---
# 6 Bibliografía

[1]Algoritmo de inversion de colores [Algoritmo](https://www.youtube.com/watch?v=w4GZPcIrvGs)

[2] Documentacion Pycuda [WEB](https://documen.tician.de/pycuda/index.html)

[3] Repositorio Pycuda [WEB](https://pypi.org/project/pycuda/)
